# Workshop: Traditional ML on Union

**By the end you'll know how to:**
- Write Union tasks and run them locally or on the cluster
- Parallelize workloads across Kubernetes pods
- Cache expensive steps, pass artifacts between tasks
- Schedule monthly workflows
- Host a Streamlit dashboard on Union

**Flyte** = the SDK you write code with.  
**Union** = the platform that runs it.

---

## Setup

```bash
uv venv && source .venv/bin/activate
uv pip install -r requirements.txt
flyte create config
```

In [2]:
import flyte
print(f"Flyte SDK: {flyte.__version__}")

Flyte SDK: 2.0.3


---

## Part 1: Fundamentals

| Concept | What it is |
|---------|------------|
| **TaskEnvironment** | Shared config — resources, image, caching |
| **@env.task** | Decorator: Python function → containerized job |
| **flyte.map()** | Parallel execution across inputs |
| **flyte.run()** | Execute on the cluster |

### Exercise 0: Hello Union

In [3]:
import flyte

env = flyte.TaskEnvironment(
    name="hello_union",
    resources=flyte.Resources(memory="250Mi"),
)


@env.task
def greet(name: str) -> str:
    return f"Hello, {name}!"


@env.task
def main(names: list[str] = ["Alice", "Bob", "Charlie", "Diana"]) -> list[str]:
    return list(flyte.map(greet, names))

In [4]:
# Run locally (no cluster needed)
result = main()
print(result)

[flyte] Running map in local mode, which will run every task sequentially.


['Hello, Alice!', 'Hello, Bob!', 'Hello, Charlie!', 'Hello, Diana!']


In [5]:
# Run on cluster
flyte.init_from_config()
run = flyte.run(main)
print(f"Run URL: {run.url}")
run.wait()

> Building 1 image...

> Building image flyte for environment hello_union

i Image us-east4-docker.pkg.dev/qxo-data-science-dev-462616/orgs/qxo/flyte:py3.12-v2.0.3 already exists, 
skipping build

✓ Built image for environment hello_union: 
us-east4-docker.pkg.dev/qxo-data-science-dev-462616/orgs/qxo/flyte:py3.12-v2.0.3

/Users/kyle/Projects/solutions-engineering/onboarding_workshop/.venv/lib/python3.12/site-packages/rich/live.py:260:
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Run URL: https://qxo.hosted.unionai.cloud/v2/domain/development/project/onboarding/runs/rqjssgrsh92fcwhqfxf4


Run 'rqjssgrsh92fcwhqfxf4' completed successfully.

**What happened:**
1. `main()` ran in a pod → called `greet()` 4 times in parallel
2. Each `greet()` call = its own pod (own container, own resources)
3. Open the **Run URL** → see the execution graph in the Union UI

### Parallel Processing

Pattern for entity resolution / batch ETL:  
**split → process in parallel → aggregate**

In [6]:
env = flyte.TaskEnvironment(
    name="parallel",
    resources=flyte.Resources(cpu="2", memory="2Gi"),
)


@env.task
def process_chunk(chunk_id: int) -> dict:
    """Each chunk gets its own pod with 2 CPU / 2Gi RAM."""
    import random
    records = random.randint(1000, 5000)
    matches = random.randint(10, 100)
    print(f"Chunk {chunk_id}: {records} records, {matches} matches")
    return {"chunk_id": chunk_id, "records": records, "matches": matches}


@env.task
def aggregate(results: list[dict]) -> dict:
    return {
        "chunks": len(results),
        "total_records": sum(r["records"] for r in results),
        "total_matches": sum(r["matches"] for r in results),
    }


@env.task
def pipeline(num_chunks: int = 10) -> dict:
    results = list(flyte.map(process_chunk, range(num_chunks)))
    return aggregate(results)

In [7]:
# 10 chunks processed in parallel, each in its own pod
run = flyte.run(pipeline, num_chunks=10)
print(f"Run URL: {run.url}")
run.wait()

> Building 1 image...

> Building image flyte for environment parallel

✓ Built image for environment parallel: 
us-east4-docker.pkg.dev/qxo-data-science-dev-462616/orgs/qxo/flyte:py3.12-v2.0.3

Run URL: https://qxo.hosted.unionai.cloud/v2/domain/development/project/onboarding/runs/rng82xd4xpshmvngnn7w


Run 'rng82xd4xpshmvngnn7w' completed successfully.

**Error handling:** `return_exceptions=True` lets you handle failures without crashing the whole fan-out.

```python
for result in flyte.map(risky_task, data, return_exceptions=True):
    if isinstance(result, Exception):
        print(f"Skipping: {result}")
    else:
        process(result)
```

---

## Part 2: ML Pipeline Concepts

### Image Building

Union builds container images for you. No Dockerfile, no registry, no `docker build`.

```python
img = flyte.Image.from_debian_base().with_pip_packages(
    "pandas", "lightgbm", "scikit-learn"
)
```

Images are cached and optimized for fast pulls on the cluster.

In [20]:
import json
import flyte.report
from flyte.io import File

img = flyte.Image.from_debian_base().with_pip_packages(
    "pandas", "pyarrow", "scikit-learn", "lightgbm", "optuna"
)

ml_env = flyte.TaskEnvironment(
    name="ml_pipeline",
    image=img,
    resources=flyte.Resources(cpu="1", memory="1Gi"),
)

### Caching

`cache="auto"` — Union skips tasks whose code + inputs haven't changed.  
Critical for monthly workflows: don't reload data just to tweak hyperparameters.

In [9]:
@ml_env.task(cache="auto")
async def load_data() -> File:
    """Only runs once. Reuses cached output on subsequent runs."""
    import numpy as np
    import pandas as pd

    np.random.seed(42)
    df = pd.DataFrame({
        "price": 100 + np.cumsum(np.random.randn(5000) * 0.5),
        "volume": np.random.exponential(1000, 5000),
        "hour": np.tile(range(24), 5000 // 24 + 1)[:5000],
        "day_of_week": np.tile(range(7), 5000 // 7 + 1)[:5000],
    })
    df.to_parquet("data.parquet", index=False)
    print(f"Loaded {len(df)} rows")
    return await File.from_local("data.parquet")

### File Artifacts

`flyte.io.File` — pass model files, data files between tasks.  
Automatically uploaded/downloaded.

In [10]:
@ml_env.task
async def train(data_file: File) -> tuple[File, str]:
    import lightgbm as lgb
    import pandas as pd
    from sklearn.metrics import mean_absolute_error
    from sklearn.model_selection import train_test_split

    path = await data_file.download()
    df = pd.read_parquet(path)
    X, y = df.drop(columns=["price"]), df["price"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

    model = lgb.LGBMRegressor(n_estimators=50, verbose=-1)
    model.fit(X_train, y_train)
    mae = mean_absolute_error(y_test, model.predict(X_test))

    import joblib
    joblib.dump(model, "model.joblib")

    metrics = {"mae": round(mae, 4), "features": list(X.columns),
               "importances": model.feature_importances_.tolist()}
    return await File.from_local("model.joblib"), json.dumps(metrics)

### Reports

`report=True` — generate rich HTML visible as a tab in the Union UI.  
Great for feature importance, SHAP values, forecast charts.

In [11]:
@ml_env.task(report=True)
async def report(metrics_json: str) -> str:
    m = json.loads(metrics_json)
    rows = sorted(zip(m["features"], m["importances"]), key=lambda x: -x[1])
    table = "".join(f"<tr><td>{f}</td><td>{i}</td></tr>" for f, i in rows)

    html = f"""
    <h2>Forecast Model Report</h2>
    <p><b>MAE:</b> {m['mae']}</p>
    <h3>Feature Importance</h3>
    <table border='1' cellpadding='6' style='border-collapse:collapse;'>
        <tr style='background:#eee;'><td><b>Feature</b></td><td><b>Importance</b></td></tr>
        {table}
    </table>
    """
    await flyte.report.replace.aio(html)
    await flyte.report.flush.aio()
    return f"MAE: {m['mae']}"

In [21]:
# Full pipeline
@ml_env.task
async def ml_pipeline() -> str:
    data_file = await load_data()           # cached after first run
    model_file, metrics = await train(data_file)
    return await report(metrics)

In [22]:
run = flyte.run(ml_pipeline)
print(f"Run URL: {run.url}")
run.wait()

> Building 1 image...

> Building image flyte for environment ml_pipeline

i Image flyte:9516a853d1954c2cd71afd7ce742f97e was not found or has expired

> Image ghcr.io/flyteorg/flyte:9516a853d1954c2cd71afd7ce742f97e not found, building...

> Submitting a new build...

...

✓ Build completed in 0:00:52

✓ Built image for environment ml_pipeline: 
us-east4-docker.pkg.dev/qxo-data-science-dev-462616/orgs/qxo/flyte:9516a853d1954c2cd71afd7ce742f97e

/Users/kyle/Projects/solutions-engineering/onboarding_workshop/.venv/lib/python3.12/site-packages/rich/live.py:260:
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Run URL: https://qxo.hosted.unionai.cloud/v2/domain/development/project/onboarding/runs/rhswqn6k44qxg9rxhm4t


Run 'rhswqn6k44qxg9rxhm4t' completed successfully.

In [14]:
# Run again — load_data hits cache, only train + report re-run
run = flyte.run(ml_pipeline)
print(f"Run URL: {run.url}")
run.wait()

Run URL: https://qxo.hosted.unionai.cloud/v2/domain/development/project/onboarding/runs/rb55x724mqhnmk76zsvx


Run 'rb55x724mqhnmk76zsvx' completed successfully.

### OOM Retry

Union detects when a pod is killed for exceeding its memory limit and raises `flyte.errors.OOMError`.  
You can catch it and retry with more resources:

In [15]:
import flyte.errors

@ml_env.task
async def memory_hungry(size_mb: int) -> str:
    """Allocate memory. Will OOM if size exceeds the pod limit."""
    data = bytearray(size_mb * 1024 * 1024)
    print(f"Allocated {size_mb}MB successfully")
    return f"Processed {size_mb}MB"


@ml_env.task
async def pipeline_with_oom_retry() -> str:
    """Try with small memory, catch OOM, retry with more."""
    mem_levels = ["256Mi", "512Mi", "1Gi"]
    for mem in mem_levels:
        try:
            return await memory_hungry.override(
                resources=flyte.Resources(cpu="1", memory=mem)
            )(400)  # try to allocate 400MB
        except flyte.errors.OOMError:
            print(f"OOM with {mem}, retrying with more memory...")
    raise RuntimeError("Failed even with max memory")

### Hyperparameter Tuning

Run multiple trials in parallel. Each trial = its own pod.

In [25]:
import asyncio
from typing import Union as UnionType

hpo_env = flyte.TaskEnvironment(
    name="hpo",
    image=img,
    resources=flyte.Resources(cpu="1", memory="1Gi"),
    cache="auto",  # env-level: all tasks cached
)


@hpo_env.task
async def run_trial(params: dict[str, UnionType[int, float]]) -> float:
    """Cached — identical params won't re-run."""
    import lightgbm as lgb
    import numpy as np
    import pandas as pd
    from sklearn.model_selection import cross_val_score

    np.random.seed(42)
    df = pd.DataFrame({
        "target": np.random.randn(2000),
        "f1": np.random.randn(2000),
        "f2": np.random.exponential(1, 2000),
    })
    model = lgb.LGBMRegressor(
        num_leaves=int(params["num_leaves"]),
        n_estimators=int(params["n_estimators"]),
        learning_rate=params["learning_rate"],
        verbose=-1,
    )
    scores = cross_val_score(model, df[["f1", "f2"]], df["target"],
                            cv=3, scoring="neg_mean_absolute_error")
    return float(-scores.mean())


@hpo_env.task
async def optimize(n_trials: int = 10, concurrency: int = 3) -> dict:
    import optuna
    study = optuna.create_study(direction="minimize")
    sem = asyncio.Semaphore(concurrency)

    async def one_trial():
        async with sem:
            t = study.ask()
            params = {
                "num_leaves": t.suggest_int("num_leaves", 10, 60),
                "n_estimators": t.suggest_int("n_estimators", 30, 150),
                "learning_rate": t.suggest_float("learning_rate", 0.01, 0.3, log=True),
            }
            score = await run_trial(params)
            study.tell(t, score)

    await asyncio.gather(*[one_trial() for _ in range(n_trials)])
    print(f"Best: {study.best_trial.params}")
    return study.best_trial.params

In [26]:
run = flyte.run(optimize, n_trials=10, concurrency=3)
print(f"Run URL: {run.url}")
run.wait()

> Building 1 image...

> Building image flyte for environment hpo

✓ Built image for environment hpo: 
us-east4-docker.pkg.dev/qxo-data-science-dev-462616/orgs/qxo/flyte:9516a853d1954c2cd71afd7ce742f97e

Run URL: https://qxo.hosted.unionai.cloud/v2/domain/development/project/onboarding/runs/rcz2j2b9dvf64lzjrxwf


Run 'rcz2j2b9dvf64lzjrxwf' completed successfully.

---

## Part 3: Production Features

### Scheduled Triggers

Run the forecasting workflow automatically on the 1st of every month:

In [18]:
from datetime import datetime

prod_env = flyte.TaskEnvironment(
    name="production",
    resources=flyte.Resources(cpu="1", memory="1Gi"),
)


@prod_env.task(
    triggers=flyte.Trigger(
        "monthly_forecast",
        flyte.Cron("0 6 1 * *"),   # 6 AM on the 1st of every month
        inputs={"run_date": flyte.TriggerTime},
    ),
    cache="auto",
)
async def monthly_forecast(run_date: datetime) -> str:
    month = run_date.strftime("%Y-%m")
    print(f"Running forecast for {month}")
    return f"Forecast done for {month}"

Activate with `flyte deploy 04_production.py`. This registers the task and starts the cron schedule.

### Domains

| Domain | Purpose |
|--------|---------|
| `development` | Experimentation, fast iteration |
| `staging` | Integration testing |
| `production` | Scheduled workflows, serving |

Each domain = separate Kubernetes namespace (isolated resources).

```bash
flyte run 02_ml_pipeline.py pipeline                      # development
flyte run --domain production 02_ml_pipeline.py pipeline   # production
```

### BigQuery

Native connector — no credential management needed (uses cluster service account):

```python
from flyteplugins.connectors.bigquery import BigQueryConfig, BigQueryTask
from flyte.io import DataFrame

load_prices = BigQueryTask(
    name="load_prices",
    inputs={"month": str},
    output_dataframe_type=DataFrame,
    plugin_config=BigQueryConfig(ProjectID="my-gcp-project"),
    query_template="SELECT * FROM prices WHERE month = '{{ .Inputs.month }}'",
)
```

### Streamlit Dashboard

Host a Streamlit app on Union — get an endpoint your team can access.  
`AppEnvironment` is like `TaskEnvironment`, but for long-running apps.

In [19]:
import flyte.app

app_env = flyte.app.AppEnvironment(
    name="forecast-dashboard",
    image=flyte.Image.from_debian_base(python_version=(3, 12)).with_pip_packages(
        "streamlit==1.41.1", "pandas", "numpy"
    ),
    args="streamlit run dashboard.py --server.port 8080",
    port=8080,
    include=["dashboard.py"],
    resources=flyte.Resources(cpu="1", memory="1Gi"),
    requires_auth=False,
)

See `05_streamlit_dashboard.py` for the full working example.  
Deploy with:

```bash
python 05_streamlit_dashboard.py
```

Union gives you a URL your team can access — no infrastructure to manage.

---

### Local Development with the TUI

You don't need the cluster to iterate. `flyte run --local --tui` gives you an interactive terminal dashboard for monitoring local runs.

```bash
# Run locally with the TUI
flyte run --local --tui 02_ml_pipeline.py pipeline

# Browse past local runs
flyte start tui
```

The TUI shows a split-screen view:
- **Left panel** — task tree with live status indicators
- **Right panel** — logs and details for the selected task

Status indicators:

| Symbol | Meaning |
|--------|---------|
| `●` | Running |
| `✓` | Completed |
| `✗` | Failed |
| `$` | Cache hit (skipped) |
| `~` | Cache miss (ran despite caching enabled) |

Keyboard shortcuts: `l` = logs tab, `d` = details tab, `q` = quit.

Flyte persists inputs and outputs locally, so you can browse previous runs with `flyte start tui` without re-executing anything. This is useful for comparing results across experiments.

**Typical dev workflow:**
1. Iterate locally with `flyte run --local --tui` — fast, no cluster needed
2. When ready, push to cluster with `flyte run` — same code, no changes

## Summary

| Concept | Use Case |
|---------|----------|
| `flyte.map()` | Parallelize batch processing, entity resolution |
| `cache="auto"` | Skip data loading on monthly reruns |
| `flyte.io.File` | Pass model artifacts between pipeline steps |
| `flyte.report` | Feature importance, forecast charts in UI |
| `.override(resources=...)` + OOM retry | Handle variable data sizes gracefully |
| `flyte.Trigger` / `flyte.Cron` | Monthly forecasting cron |
| `flyte.Image` | No Docker, just list pip packages |
| `AppEnvironment` | Streamlit dashboards hosted on Union |
| Domains | Dev/staging/prod isolation |
| BigQuery connector | Native GCP integration |

## Next Steps

1. Port your forecasting workflow using patterns from exercises 2 + 4
2. Set up the monthly trigger
3. **Workshop 2** — Agentic AI on Union (LangGraph, real-time inference, app serving)